# Week 23, Day 2: Regular expressions II

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w023d2.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
import re

m = re.search(r"(\d{2})/(\d{2})/(\d{4})", "Due 31/10/2026, paid 02/11/2026")
print(m.group(0))
print(m.group(1), m.group(2), m.group(3))
print(m.groups())

In [ ]:
import re

pattern = r"(?P<day>\d{2})/(?P<month>\d{2})/(?P<year>\d{4})"
m = re.search(pattern, "Due 31/10/2026")
print(m.group("year"), m.groupdict())
iso = f"{m['year']}-{m['month']}-{m['day']}"        # m["name"] works too
print(iso)

In [ ]:
import re

text = "width=80 height=24 depth=3"
print(re.findall(r"\w+=\d+", text))
print(re.findall(r"(\w+)=\d+", text))
print(re.findall(r"(\w+)=(\d+)", text))
print(dict(re.findall(r"(\w+)=(\d+)", text)))

In [ ]:
import re

text = "Monday, Tuesday and Sunday"
print(re.findall(r"\b(Mon|Tues|Sun)day\b", text))     # captures just the first part
print(re.findall(r"\b(?:Mon|Tues|Sun)day\b", text))   # whole words

*Interactive figure on the website: Named groups show up by name. Try the presets, then write your own..*

In [ ]:
import re

print(re.sub(r"\s+", " ", "too    many   spaces"))
print(re.sub(r"(\d{2})/(\d{2})/(\d{4})", r"\3-\2-\1", "Due 31/10/2026, paid 02/11/2026"))
print(re.sub(r"(?P<first>\w+) (?P<last>\w+)", r"\g<last>, \g<first>", "Ada Lovelace"))
print(re.sub(r"\d", "#", "Card 4929 1234", count=4))

In [ ]:
import re

def double(m):
    return str(int(m.group()) * 2)

print(re.sub(r"\d+", double, "3 eggs, 250 g flour, 2 cups milk"))
print(re.sub(r"\b\w", lambda m: m.group().upper(), "the lord of the rings"))

In [ ]:
import re

print(re.split(r"[,;]\s*", "apples, pears;plums,  figs"))
print(re.split(r"\s*(?:and|or)\s*", "tea and cake or biscuits"))
print(re.split(r"(\d+)", "a1b22c"))           # with a group, the separators are kept

In [ ]:
import re

POSTCODE = re.compile(r"""
    \b
    (?P<area>[A-Z]{1,2})       # 1 or 2 letters
    (?P<district>\d[A-Z\d]?)   # a digit, maybe a letter or digit
    \ 
    (?P<sector>\d)             # inward code: a digit...
    (?P<unit>[A-Z]{2})         # ...and two letters
    \b
""", re.VERBOSE)

m = POSTCODE.search("Send it to SW1A 2AA please")
print(m.groupdict())

In [ ]:
import re

html = "<b>bold</b> and <i>italic</i>"
print(re.findall(r"<.+>", html))     # greedy: one huge match
print(re.findall(r"<.+?>", html))    # lazy: each tag
print(re.findall(r'"(.*?)"', 'say "hi" then "bye"'))

In [ ]:
import re

def strong(password):
    return re.fullmatch(r"(?=.*[a-z])(?=.*[A-Z])(?=.*\d).{10,}", password) is not None

print([strong(p) for p in ["Password123", "password123", "Pass1", "Tr1ckyHorse!"]])
print(re.findall(r"\d+(?= pounds)", "5 pounds, 7 euros, 12 pounds"))

> **Key idea.** Groups `(...)` capture parts of a match: `m.group(1)`, `m.groups()`, or named with `(?P<name>...)` and `m.groupdict()`. `findall` returns groups when there are any; use `(?:...)` to group without capturing. `re.sub` replaces, with `\1` or a function; `re.split` splits on a pattern. Add `?` to make a quantifier lazy; `(?=...)` looks ahead. Use string methods or a proper parser when they fit better.

### Exercise: Pull out the parts

Write `parse_time(text)` that finds the first time like `14:05` or `9:30 pm` in text and returns a tuple `(hours, minutes)` as whole numbers in 24-hour form (`9:30 pm` is `(21, 30)`, `12:15 am` is `(0, 15)`, `12:15 pm` is `(12, 15)`), or `None`. Use one pattern with named groups `hour`, `minute` and an optional `ampm` (`am` or `pm`, any case, after an optional space).

In [ ]:
import re

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    assert parse_time(\"Meet at 14:05 today\") == (14, 5)\n    assert parse_time(\"Train 9:30 pm\") == (21, 30) and parse_time(\"9:30AM\") == (9, 30)\n    assert parse_time(\"12:15 am\") == (0, 15) and parse_time(\"12:15 pm\") == (12, 15)\n    assert parse_time(\"no time\") is None")

### Exercise: Read settings

A settings file has lines `key = value`, with optional spaces around `=`, comments starting with `#`, and blank lines. Write `read_settings(text)` returning a dictionary of the settings, using one `re.findall` with two groups and `re.MULTILINE`. Keys are letters, digits, underscores and dots; values run to the end of the line, with trailing spaces removed, and may be empty.

In [ ]:
import re

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    text = \"\"\"# app settings\nname = Python Atlas\ndebug=true\ndb.host =  localhost\n  # indented comment\nempty =\n\"\"\"\n    assert read_settings(text) == {\"name\": \"Python Atlas\", \"debug\": \"true\", \"db.host\": \"localhost\", \"empty\": \"\"}")

### Exercise: Rearrange with sub

Using `re.sub` with backreferences or named groups:

- `to_iso(text)`: change every British date `DD/MM/YYYY` in a text to `YYYY-MM-DD`;
- `mask_card(text)`: replace all but the last 4 digits of every 16-digit card number written as four groups of four (separated by spaces or hyphens) with `*`, keeping the separators: `4929 1234 5678 9012` becomes `**** **** **** 9012`.

In [ ]:
import re

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    assert to_iso(\"From 31/10/2026 to 02/11/2026.\") == \"From 2026-10-31 to 2026-11-02.\"\n    assert mask_card(\"Card 4929 1234 5678 9012 ok\") == \"Card **** **** **** 9012 ok\"\n    assert mask_card(\"4929-1234-5678-9012\") == \"****-****-****-9012\"\n    assert mask_card(\"ref 1234\") == \"ref 1234\"")

### Exercise: Replace with a function

Write `convert_temps(text)` that finds every Fahrenheit temperature written like `72F` or `-4 F` (a whole or decimal number, an optional space, then `F` as a whole word) and replaces it with the Celsius value rounded to 1 decimal place followed by `C`, using `re.sub` with a function.

In [ ]:
import re

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    assert convert_temps(\"Today 72F, tonight 50 F.\") == \"Today 22.2C, tonight 10.0C.\"\n    assert convert_temps(\"Freezing: 32F; very cold -4 F\") == \"Freezing: 0.0C; very cold -20.0C\"\n    assert convert_temps(\"Fahrenheit 451 by Bradbury\") == \"Fahrenheit 451 by Bradbury\"")

### Exercise: Tokenise a sum

Write `tokens(expr)` that splits an arithmetic expression into numbers (whole or decimal), operators `+ - * /` and brackets, dropping spaces, using `re.findall` with alternatives. Then `evaluate(expr)` that checks every character was used (raise `ValueError` if the expression contains anything else) and returns the result of `eval` on the joined tokens. (A real calculator would parse the tokens itself; Phase 3 does that.)

In [ ]:
import re

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    assert tokens(\"3 + 4.5*(2 - 1)\") == [\"3\", \"+\", \"4.5\", \"*\", \"(\", \"2\", \"-\", \"1\", \")\"]\n    assert tokens(\"\") == []\n    assert evaluate(\"2*(3+4)\") == 14 and evaluate(\"10 / 4\") == 2.5\n    try:\n        evaluate(\"2 + __import__('os')\")\n        assert False, \"should reject\"\n    except ValueError:\n        pass")

### Bug hunt: Too greedy

This should print each quoted phrase separately, `['red', 'blue']`, but the greedy `.*` swallows everything between the first and last quote. Fix it.

In [ ]:
import re
print(re.findall(r'"(.*)"', 'pick "red" or "blue"'))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"fixed\"\n    expect_output(\"['red', 'blue']\")")

### Bug hunt: Unwanted capture

This should list the full prices with their currency, `['£5', '$12', '€3']`, but `findall` returns only the currency symbols because of the group used for the alternatives. Fix the pattern.

In [ ]:
import re
print(re.findall(r"(£|\$|€)\d+", "Tea £5, lunch $12, coffee €3"))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"fixed\"\n    expect_output(\"['\u00a35', '$12', '\u20ac3']\")")

### Puzzle: Swap names

Arrange the lines so the program turns "First Last" names into "LAST, First". One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them

Work these out without running them.

In [ ]:
import re

a = ___   # re.findall(r"(\w)(\d)", "a1 b2 c")
b = ___   # re.sub(r"(\w+)@(\w+)", r"\2 at \1", "ada@home")
c = ___   # re.split(r"\s*,\s*", "a , b,c ,d")
d = ___   # re.findall(r"<(.+?)>", "<a><b>")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert a == [(\"a\", \"1\"), (\"b\", \"2\")] and b == \"home at ada\" and c == [\"a\", \"b\", \"c\", \"d\"] and d == [\"a\", \"b\"]")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a log report

Web server logs have lines like:

```
203.0.113.9 - - [31/Oct/2026:14:02:11 +0000] "GET /days/w023d2.html HTTP/1.1" 200 5123
```

Write `parse_line(line)` returning a dictionary with `"ip"`, `"time"` (a `datetime`, using `strptime` with `"%d/%b/%Y:%H:%M:%S"`), `"method"`, `"path"`, `"status"` (int) and `"bytes"` (int; a `-` means 0), or `None` for a line that doesn't match. Use one compiled pattern in `re.VERBOSE` mode with named groups. Then `report(log_text)` returning a dictionary with `"requests"` (lines parsed), `"errors"` (status 400 or more), `"top_path"` (the most requested path, the alphabetically first if tied), and `"bytes"` (the total).

In [ ]:
import re
from datetime import datetime

In [ ]:
# Check: run the exercise cell above first, then this one.
check("LOG = \"\"\"203.0.113.9 - - [31/Oct/2026:14:02:11 +0000] \"GET /days/w023d2.html HTTP/1.1\" 200 5123\n198.51.100.4 - - [31/Oct/2026:14:02:15 +0000] \"GET /index.html HTTP/1.1\" 200 8900\ngarbage line\n203.0.113.9 - - [31/Oct/2026:14:03:01 +0000] \"POST /api/save HTTP/1.1\" 500 -\n192.0.2.1 - - [31/Oct/2026:14:05:42 +0000] \"GET /days/w023d2.html HTTP/1.1\" 404 210\n\"\"\"\ndef test_parse():\n    \"parse_line\"\n    from datetime import datetime\n    p = parse_line(LOG.splitlines()[0])\n    assert p == {\"ip\": \"203.0.113.9\", \"time\": datetime(2026, 10, 31, 14, 2, 11), \"method\": \"GET\", \"path\": \"/days/w023d2.html\", \"status\": 200, \"bytes\": 5123}\n    assert parse_line(\"garbage line\") is None and parse_line(LOG.splitlines()[3])[\"bytes\"] == 0\ndef test_report():\n    \"report\"\n    assert report(LOG) == {\"requests\": 4, \"errors\": 2, \"top_path\": \"/days/w023d2.html\", \"bytes\": 14233}")

## Warm-up

Round brackets in a pattern make a **group**. The match as a whole is still group 0, but now each bracketed part is captured on its own, ready to use: day, month, year.

## Groups

Every pair of round brackets is numbered by its opening bracket, from the left, starting at 1:

- `m.group(1)` is the text captured by the first group, `m.group(2)` the second, and so on;
- `m.groups()` gives all of them as a tuple;
- a group that didn't take part in the match (inside an optional `?`, say) gives `None`.

**Named groups** are clearer: `(?P<name>...)`. Read them with `m.group("name")` or all at once with `m.groupdict()`:

Groups also let a quantifier apply to more than one character: `(ha)+` matches "hahaha", and `(\d{3}-)?\d{4}` makes a whole prefix optional.

**`findall` with groups** behaves differently, and catches everyone out once:

- no groups: a list of the whole matches;
- one group: a list of **just that group's** text;
- several groups: a list of **tuples**, one item per group.

When you need brackets only for grouping (for `|` or a quantifier) but don't want to capture, use a **non-capturing group** `(?:...)`. Then `findall` ignores it:

Try the groups here; the table lists each match's groups:

## Substitution

`re.sub(pattern, replacement, text)` replaces every match. The replacement can refer to groups with `\1`, `\2` (or `\g<name>` for named groups), which lets you **rearrange** text:

`count=` limits the number of replacements; `re.subn` returns the new text **and** how many replacements it made.

For replacements that need **working out**, pass a **function** instead of a string. It's called with each match object and returns the replacement text:

## Splitting and compiling

`re.split(pattern, text)` splits wherever the pattern matches, which handles messy separators that `str.split` can't:

**Compiling** a pattern with `re.compile` gives a pattern object with the same methods (`.search`, `.findall`, `.sub`, ...). It's tidy when a pattern is used many times, and gives it a name.

**Verbose mode** (`re.VERBOSE`, or `re.X`) lets you spread a pattern over several lines with comments: spaces and `#` comments in the pattern are ignored (write `\ ` or `[ ]` for a real space).

## Greedy, lazy and lookahead

Quantifiers are **greedy**: they take as much as they can. Adding `?` after a quantifier makes it **lazy**: as little as it can.

**Lookahead** checks what comes next without including it in the match: `(?=...)` means "followed by", `(?!...)` "not followed by". (Lookbehind, `(?<=...)` and `(?<!...)`, checks what came before.) A classic use is checking several rules at once:

Each `(?=.*X)` looks ahead from the start for an X without moving, so all the conditions apply to the same text.

## When not to use a regex

- **Simple jobs have simple tools.** `"@" in s`, `s.startswith("http")`, `s.split(",")` and `s.replace("a", "b")` are clearer and faster.
- **Nested structures**: HTML, JSON and programming languages can nest brackets to any depth, which regular expressions can't track. Use a real parser: `json` (Day 3), `csv` (Day 4), `html.parser`, or `ast` for Python.
- **Unreadable patterns**: if you can't explain it, use `re.VERBOSE` with comments, or split the job into steps.

As programmers joke: "Some people, when confronted with a problem, think 'I know, I'll use regular expressions.' Now they have two problems." Use them for what they're good at: finding and pulling apart text with a regular shape.

## Practice

## Quiz

## Daily challenge

## Before you go

Find a text file on your computer with some structure (a list of contacts, a log, an export from an app) and write three regex questions about it: find something, pull out parts of something, and rewrite something. Tomorrow: JSON, the format almost every web service uses to exchange data.